# Unit Testing - Analisis Okupansi Ruangan
Notebook ini memuat setup awal untuk membaca data dari `dataset/data_.csv` dan beberapa cell di bawahnya berisi *unit test* terpisah untuk masing-masing fungsi dan class.

In [18]:
import unittest
import os
from Models import SensorReading
from Repository import SensorRepository
from Analyzer import CO2Analyzer, ThermodynamicAnalyzer, EnergyEfficiencyAnalyzer

# Setup Global: Memuat data dari CSV asli agar bisa dipakai di seluruh cell testing
repo = SensorRepository()
file_path = 'dataset/data_.csv'

try:
    total_data = repo.load_csv(file_path)
    data_sensor = repo.get_all()
    print(f"[BERHASIL] {total_data} baris data dimuat dari {file_path}")
except FileNotFoundError:
    print(f"[GAGAL] File {file_path} tidak ditemukan. Pastikan path sesuai.")

[BERHASIL] 8143 baris data dimuat dari dataset/data_.csv


### 1. Test Validasi Model (Models.py)

In [19]:
class TestModelValidation(unittest.TestCase):
    def test_sensor_reading_validation(self):
        """Memastikan error ValueError muncul saat inisiasi data tidak valid"""
        # Test 1: Kelembapan (Humidity) lebih dari 100%
        with self.assertRaises(ValueError):
            SensorReading("2026-06-09", 25.0, 150.0, 100, 500.0, 0.004, 0)
        
        # Test 2: CO2 di bawah ambang batas (< 400)
        with self.assertRaises(ValueError):
            SensorReading("2026-06-09", 25.0, 50.0, 100, 300.0, 0.004, 0)

# Eksekutor spesifik hanya untuk cell ini
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestModelValidation)
unittest.TextTestRunner(verbosity=2).run(suite)

test_sensor_reading_validation (__main__.TestModelValidation.test_sensor_reading_validation)
Memastikan error ValueError muncul saat inisiasi data tidak valid ... ok

----------------------------------------------------------------------
Ran 1 test in 0.003s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

### 2. Test Fungsi is_occupied (Repository.py)

In [20]:
class TestRepositoryLogic(unittest.TestCase):
    def test_get_occupied_filters_correctly(self):
        """Memastikan get_occupied() hanya mereturn data yang Occupancy-nya 1"""
        occupied_data = repo.get_occupied()
        
        # Cek apakah setiap baris hasil filter benar-benar occupied
        for row in occupied_data:
            self.assertTrue(row.is_occupied(), "Ada data yang terfilter tapi is_occupied() bernilai False")

# Eksekutor spesifik hanya untuk cell ini
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestRepositoryLogic)
unittest.TextTestRunner(verbosity=2).run(suite)

test_get_occupied_filters_correctly (__main__.TestRepositoryLogic.test_get_occupied_filters_correctly)
Memastikan get_occupied() hanya mereturn data yang Occupancy-nya 1 ... ok

----------------------------------------------------------------------
Ran 1 test in 0.004s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

### 3. Test CO2 Analyzer (Analyzer.py)

In [21]:
class TestCO2Analyzer(unittest.TestCase):
    def test_co2_analysis_structure(self):
        """Menguji output dari CO2Analyzer terhadap struktur Dictionary yang diharapkan"""
        analyzer = CO2Analyzer(data_sensor)
        result = analyzer.analyze()
        
        # Cek ketersediaan key pada dictionary return
        self.assertIn("overall_occupancy", result)
        self.assertIn("average_co2_when_occupied", result)
        self.assertIn("status", result)
        
        # Cek apakah status sesuai dengan opsi valid yang diset di kodingan
        self.assertIn(result["status"], ["Ventilasi Buruk", "Ventilasi Baik"])

# Eksekutor spesifik hanya untuk cell ini
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCO2Analyzer)
unittest.TextTestRunner(verbosity=2).run(suite)

test_co2_analysis_structure (__main__.TestCO2Analyzer.test_co2_analysis_structure)
Menguji output dari CO2Analyzer terhadap struktur Dictionary yang diharapkan ... ok

----------------------------------------------------------------------
Ran 1 test in 0.004s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

### 4. Test Thermodynamic Analyzer (Analyzer.py)

In [22]:
class TestThermodynamicAnalyzer(unittest.TestCase):
    def test_thermodynamic_calculations(self):
        """Menguji kalkulasi Max/Min Temperature dan rata-rata Humidity"""
        analyzer = ThermodynamicAnalyzer(data_sensor)
        result = analyzer.analyze()
        
        # Pastikan dictionary tidak kosong dan punya structure yang benar
        self.assertTrue(len(result) > 0)
        self.assertIn("temperature_range", result)
        
        # Max tidak boleh lebih kecil dari Min
        t_max = result["temperature_range"]["max"]
        t_min = result["temperature_range"]["min"]
        self.assertGreaterEqual(t_max, t_min, "Max temperature harus >= min temperature")

# Eksekutor spesifik hanya untuk cell ini
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestThermodynamicAnalyzer)
unittest.TextTestRunner(verbosity=2).run(suite)

test_thermodynamic_calculations (__main__.TestThermodynamicAnalyzer.test_thermodynamic_calculations)
Menguji kalkulasi Max/Min Temperature dan rata-rata Humidity ... ok

----------------------------------------------------------------------
Ran 1 test in 0.004s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

### 5. Test Energy Efficiency Analyzer (Analyzer.py)

In [23]:
class TestEnergyEfficiencyAnalyzer(unittest.TestCase):
    def test_energy_efficiency_waste_logic(self):
        """Menguji perhitungan pemborosan energi dan keakuratan matematis"""
        analyzer = EnergyEfficiencyAnalyzer(data_sensor)
        result = analyzer.analyze()
        
        waste_percentage = result.get("waste_percentage", 0)
        
        # Persentase tidak mungkin di bawah 0 atau di atas 100
        self.assertGreaterEqual(waste_percentage, 0.0)
        self.assertLessEqual(waste_percentage, 100.0)
        
        # Cek tipe data kembalian string untuk rekomendasi dan status
        self.assertIsInstance(result["status"], str)
        self.assertIsInstance(result["recommendation"], str)

# Eksekutor spesifik hanya untuk cell ini
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestEnergyEfficiencyAnalyzer)
unittest.TextTestRunner(verbosity=2).run(suite)

test_energy_efficiency_waste_logic (__main__.TestEnergyEfficiencyAnalyzer.test_energy_efficiency_waste_logic)
Menguji perhitungan pemborosan energi dan keakuratan matematis ... ok

----------------------------------------------------------------------
Ran 1 test in 0.006s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

### 6. TestRepositoryAdditional

In [24]:
class TestRepositoryAdditional(unittest.TestCase):
    def test_get_all_returns_list(self):
        """Memastikan get_all() mengembalikan list"""
        data = repo.get_all()
        self.assertIsInstance(data, list)
    
    def test_get_all_not_empty_after_load(self):
        """Setelah load, get_all() tidak boleh kosong"""
        self.assertGreater(len(repo.get_all()), 0)
    
    def test_get_occupied_returns_only_occupied(self):
        """get_occupied() hanya mengembalikan data dengan occupancy=1"""
        occupied = repo.get_occupied()
        for row in occupied:
            self.assertEqual(row.occupancy, 1)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestRepositoryAdditional)
unittest.TextTestRunner(verbosity=2).run(suite)

test_get_all_not_empty_after_load (__main__.TestRepositoryAdditional.test_get_all_not_empty_after_load)
Setelah load, get_all() tidak boleh kosong ... ok
test_get_all_returns_list (__main__.TestRepositoryAdditional.test_get_all_returns_list)
Memastikan get_all() mengembalikan list ... ok
test_get_occupied_returns_only_occupied (__main__.TestRepositoryAdditional.test_get_occupied_returns_only_occupied)
get_occupied() hanya mengembalikan data dengan occupancy=1 ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.008s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

### 7. TestBaseAnalyzerSummary

In [25]:
class TestBaseAnalyzerSummary(unittest.TestCase):
    def test_get_summary_has_correct_keys(self):
        """get_summary() harus punya key yang benar"""
        analyzer = CO2Analyzer(data_sensor)
        summary = analyzer.get_summary()
        
        expected_keys = ["total_records", "occupied_count", "occupancy_rate"]
        for key in expected_keys:
            self.assertIn(key, summary)
    
    def test_occupancy_rate_between_0_and_1(self):
        """occupancy_rate harus antara 0 dan 1"""
        analyzer = CO2Analyzer(data_sensor)
        rate = analyzer.get_summary()["occupancy_rate"]
        self.assertGreaterEqual(rate, 0.0)
        self.assertLessEqual(rate, 1.0)
    
    def test_occupied_count_not_exceed_total(self):
        """occupied_count tidak boleh melebihi total_records"""
        analyzer = CO2Analyzer(data_sensor)
        summary = analyzer.get_summary()
        self.assertLessEqual(summary["occupied_count"], summary["total_records"])

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestBaseAnalyzerSummary)
unittest.TextTestRunner(verbosity=2).run(suite)

test_get_summary_has_correct_keys (__main__.TestBaseAnalyzerSummary.test_get_summary_has_correct_keys)
get_summary() harus punya key yang benar ... ok
test_occupancy_rate_between_0_and_1 (__main__.TestBaseAnalyzerSummary.test_occupancy_rate_between_0_and_1)
occupancy_rate harus antara 0 dan 1 ... ok
test_occupied_count_not_exceed_total (__main__.TestBaseAnalyzerSummary.test_occupied_count_not_exceed_total)
occupied_count tidak boleh melebihi total_records ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.014s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

### 8. TestSensorDataIntegrity

In [26]:
class TestSensorDataIntegrity(unittest.TestCase):
    def test_temperature_in_reasonable_range(self):
        """Suhu harus dalam rentang wajar (10-40°C untuk ruangan)"""
        for d in data_sensor:
            self.assertGreaterEqual(d.temperature, 10.0)
            self.assertLessEqual(d.temperature, 40.0)
    
    def test_light_non_negative(self):
        """Intensitas cahaya tidak boleh negatif"""
        for d in data_sensor:
            self.assertGreaterEqual(d.light, 0)
    
    def test_co2_in_reasonable_range(self):
        """CO2 dalam rentang wajar (300-5000 ppm)"""
        for d in data_sensor:
            self.assertGreaterEqual(d.co_2, 300)
            self.assertLessEqual(d.co_2, 5000)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestSensorDataIntegrity)
unittest.TextTestRunner(verbosity=2).run(suite)

test_co2_in_reasonable_range (__main__.TestSensorDataIntegrity.test_co2_in_reasonable_range)
CO2 dalam rentang wajar (300-5000 ppm) ... ok
test_light_non_negative (__main__.TestSensorDataIntegrity.test_light_non_negative)
Intensitas cahaya tidak boleh negatif ... ok
test_temperature_in_reasonable_range (__main__.TestSensorDataIntegrity.test_temperature_in_reasonable_range)
Suhu harus dalam rentang wajar (10-40°C untuk ruangan) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.015s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

### 9. TestEmptyDataEdgeCases

In [27]:
from Analyzer import BaseIoTAnalyzer

class TestEmptyDataEdgeCases(unittest.TestCase):
    def test_empty_data_for_co2_analyzer(self):
        """CO2Analyzer dengan data kosong harus handle dengan baik"""
        analyzer = CO2Analyzer([])
        result = analyzer.analyze()
        self.assertEqual(result["average_co2_when_occupied"], 0.0)
        self.assertEqual(result["status"], "Ventilasi Baik")
    
    def test_empty_data_for_thermodynamic(self):
        """ThermodynamicAnalyzer dengan data kosong harus return {}"""
        analyzer = ThermodynamicAnalyzer([])
        result = analyzer.analyze()
        self.assertEqual(result, {})
    
    def test_empty_data_summary(self):
        """get_summary() dengan data kosong tidak boleh error"""
        class DummyAnalyzer(BaseIoTAnalyzer):
            def analyze(self):
                return {}
        
        analyzer = DummyAnalyzer([])
        summary = analyzer.get_summary()
        self.assertEqual(summary["total_records"], 0)
        self.assertEqual(summary["occupancy_rate"], 0.0)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestEmptyDataEdgeCases)
unittest.TextTestRunner(verbosity=2).run(suite)

test_empty_data_for_co2_analyzer (__main__.TestEmptyDataEdgeCases.test_empty_data_for_co2_analyzer)
CO2Analyzer dengan data kosong harus handle dengan baik ... ok
test_empty_data_for_thermodynamic (__main__.TestEmptyDataEdgeCases.test_empty_data_for_thermodynamic)
ThermodynamicAnalyzer dengan data kosong harus return {} ... ok
test_empty_data_summary (__main__.TestEmptyDataEdgeCases.test_empty_data_summary)
get_summary() dengan data kosong tidak boleh error ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.009s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

### 10. TestOccupiedVsVacant

In [28]:
class TestOccupiedVsVacant(unittest.TestCase):
    def test_co2_higher_when_occupied(self):
        """Rata-rata CO2 saat terisi harus lebih tinggi dari saat kosong"""
        occupied = [d for d in data_sensor if d.is_occupied()]
        vacant = [d for d in data_sensor if not d.is_occupied()]
        
        avg_co2_occupied = sum(d.co_2 for d in occupied) / len(occupied)
        avg_co2_vacant = sum(d.co_2 for d in vacant) / len(vacant)
        
        self.assertGreater(avg_co2_occupied, avg_co2_vacant)
        print(f"\n   📊 CO2 (Terisi): {avg_co2_occupied:.1f} ppm")
        print(f"   📊 CO2 (Kosong): {avg_co2_vacant:.1f} ppm")
    
    def test_light_higher_when_occupied(self):
        """Rata-rata cahaya saat terisi harus lebih tinggi"""
        occupied = [d for d in data_sensor if d.is_occupied()]
        vacant = [d for d in data_sensor if not d.is_occupied()]
        
        avg_light_occupied = sum(d.light for d in occupied) / len(occupied)
        avg_light_vacant = sum(d.light for d in vacant) / len(vacant)
        
        self.assertGreater(avg_light_occupied, avg_light_vacant)
        print(f"   💡 Cahaya (Terisi): {avg_light_occupied:.1f} lux")
        print(f"   💡 Cahaya (Kosong): {avg_light_vacant:.1f} lux")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestOccupiedVsVacant)
unittest.TextTestRunner(verbosity=2).run(suite)

test_co2_higher_when_occupied (__main__.TestOccupiedVsVacant.test_co2_higher_when_occupied)
Rata-rata CO2 saat terisi harus lebih tinggi dari saat kosong ... ok
test_light_higher_when_occupied (__main__.TestOccupiedVsVacant.test_light_higher_when_occupied)
Rata-rata cahaya saat terisi harus lebih tinggi ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.011s

OK



   📊 CO2 (Terisi): 1037.7 ppm
   📊 CO2 (Kosong): 490.3 ppm
   💡 Cahaya (Terisi): 459.9 lux
   💡 Cahaya (Kosong): 27.8 lux


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

### 11. TestDataConsistency

In [29]:
class TestDataConsistency(unittest.TestCase):
    def test_no_missing_values(self):
        """Pastikan tidak ada nilai yang missing/hilang"""
        for d in data_sensor:
            self.assertIsNotNone(d.timestamp)
            self.assertIsNotNone(d.temperature)
            self.assertIsNotNone(d.humidity)
            self.assertIsNotNone(d.light)
            self.assertIsNotNone(d.co_2)
    
    def test_occupancy_boolean_consistency(self):
        """is_occupied() harus konsisten dengan nilai occupancy"""
        for d in data_sensor:
            if d.occupancy == 1:
                self.assertTrue(d.is_occupied())
            else:
                self.assertFalse(d.is_occupied())

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestDataConsistency)
unittest.TextTestRunner(verbosity=2).run(suite)

test_no_missing_values (__main__.TestDataConsistency.test_no_missing_values)
Pastikan tidak ada nilai yang missing/hilang ... ok
test_occupancy_boolean_consistency (__main__.TestDataConsistency.test_occupancy_boolean_consistency)
is_occupied() harus konsisten dengan nilai occupancy ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.011s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>